In [1]:
import os
import numpy as np, pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

spark = SparkSession.builder.master("local[*]").appName("ews-records").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

cat_cols = ["code_module", "gender", "region", "highest_education",
            "imd_band", "age_band", "disability"]
meta = set(cat_cols + ["id_student", "code_presentation", "at_risk", "split"])
THRESH = {28: 0.38, 56: 0.37, 84: 0.36}       # chosen earlier on validation data
THEMES = ["engagement", "assessments", "background", "course"]

def group_of(n):
    if n.startswith("code_module"):
        return "course"
    if n.startswith(("gender", "region", "highest_education", "imd_band", "age_band",
                     "disability", "studied_credits", "num_of_prev_attempts")):
        return "background"
    if n in ("no_submission", "avg_score", "n_submitted", "n_missed", "n_late", "n_due"):
        return "assessments"
    return "engagement"

def build_records(cutoff):
    df = spark.read.parquet(f"../data/processed/features_day{cutoff}.parquet")
    num_cols = [c for c in df.columns if c not in meta]
    idx = StringIndexer(inputCols=cat_cols, outputCols=[c + "_i" for c in cat_cols], handleInvalid="keep")
    ohe = OneHotEncoder(inputCols=[c + "_i" for c in cat_cols], outputCols=[c + "_v" for c in cat_cols])
    vec = VectorAssembler(inputCols=num_cols + [c + "_v" for c in cat_cols], outputCol="features")
    model = Pipeline(stages=[idx, ohe, vec, LogisticRegression(labelCol="at_risk", maxIter=100)]) \
            .fit(df.filter("split = 'train'"))
    scored = model.transform(df)
    coef = model.stages[-1].coefficients.toArray()

    attrs = scored.schema["features"].metadata["ml_attr"]["attrs"]
    names = [None] * len(coef)
    for group in attrs.values():
        for a in group:
            names[a["idx"]] = a["name"]

    pdf = scored.select("id_student", "code_module", "code_presentation", "split", "at_risk",
                        "total_clicks", "active_days", "n_submitted", "n_due", "n_late",
                        vector_to_array("features").alias("x"),
                        vector_to_array("probability")[1].alias("risk")).toPandas()
    X = np.vstack(pdf.x.values)
    train = (pdf.split == "train").values
    mu = X[train].mean(axis=0)
    contrib = (X - mu) * coef

    # check: the contributions must add up to the model's own score
    base = model.stages[-1].intercept + coef @ mu
    logit = np.log(pdf.risk / (1 - pdf.risk))
    print(f"day {cutoff}: max error {np.abs(base + contrib.sum(axis=1) - logit).max():.1e}")

    groups = np.array([group_of(n) for n in names])
    G = pd.DataFrame({g: contrib[:, groups == g].sum(axis=1) for g in THEMES})
    avg = pdf[train][["total_clicks", "active_days", "n_submitted", "n_due", "n_late"]].mean()

    recs = []
    for k, r in enumerate(pdf.itertuples(index=False)):
        reasons = []
        for theme in ["engagement", "assessments"]:          # behaviour only
            imp = G.at[k, theme]
            if imp > 0:
                if theme == "engagement":
                    txt = (f"{r.total_clicks:.0f} clicks over {r.active_days:.0f} active days "
                           f"(typical student: {avg.total_clicks:.0f} clicks, {avg.active_days:.0f} days)")
                else:
                    txt = (f"{r.n_submitted:.0f} of {r.n_due:.0f} assessments submitted, "
                           f"{r.n_late:.0f} late (typical student: {avg.n_submitted:.1f} submitted)")
                reasons.append({"theme": theme, "impact": round(float(imp), 3), "text": txt})
        reasons.sort(key=lambda d: -d["impact"])
        recs.append({
            "id_student": int(r.id_student), "module": r.code_module,
            "presentation": r.code_presentation, "cutoff_day": cutoff, "split": r.split,
            "risk_score": round(float(r.risk), 4), "flagged": bool(r.risk >= THRESH[cutoff]),
            "actual_at_risk": int(r.at_risk),
            "theme_impacts": {t: round(float(G.at[k, t]), 3) for t in THEMES},
            "reasons": reasons})
    return recs, G, pdf

all_recs = []
for cutoff in [28, 56, 84]:
    recs, G, pdf = build_records(cutoff)
    all_recs += recs
    fl = (pdf.risk >= THRESH[cutoff]).values
    print(f"   {len(recs)} students, {fl.sum()} flagged ({fl.mean():.1%})")
    print("   average theme push among flagged:", G[fl].mean().round(3).to_dict())

os.makedirs("../data/processed", exist_ok=True)
pd.DataFrame(all_recs).to_json("../data/processed/records.jsonl", orient="records", lines=True)
print("saved", len(all_recs), "records")

day 28: max error 7.6e-14
   27538 students, 14685 flagged (53.3%)
   average theme push among flagged: {'engagement': 0.33, 'assessments': 0.331, 'background': 0.191, 'course': 0.058}
day 56: max error 5.0e-12
   26522 students, 12717 flagged (47.9%)
   average theme push among flagged: {'engagement': 0.38, 'assessments': 0.809, 'background': 0.157, 'course': 0.007}
day 84: max error 1.1e-12
   25724 students, 11303 flagged (43.9%)
   average theme push among flagged: {'engagement': 0.38, 'assessments': 1.161, 'background': 0.154, 'course': 0.05}
saved 79784 records
